# 04 FHIR Encounter Silver — Silver Layer

Read the `encounter` Bronze Delta table, parse the original FHIR JSON, clean and deduplicate records, save the Silver table, and maintain SCD Type 2 history.

**Prerequisite:** Run the matching `Encounter` ingestion notebook first and attach `FHIR_Lakehouse`.

**Bronze design:** The Bronze table stores the complete FHIR resource in `resource_json` plus extraction/load metadata. This notebook parses the JSON before accessing resource fields.

In [1]:
from pyspark.sql.functions import (
    col, lit, trim, lower, when, row_number,
    sha2, coalesce, from_json, get_json_object,
    current_timestamp
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# READ BRONZE DATA
bronze_encounter_df = spark.read.table("bronze.encounter")

print("Bronze Encounter loaded!")
print("Bronze records:", bronze_encounter_df.count())
print("Bronze columns:", bronze_encounter_df.columns)

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 3, Finished, Available, Finished, False)

Bronze Encounter loaded!
Bronze records: 2005
Bronze columns: ['appointment', 'class', 'extension', 'hospitalization', 'id', 'identifier', 'length', 'location', 'meta', 'participant', 'period', 'priority', 'reasonCode', 'resourceType', 'serviceProvider', 'serviceType', 'status', 'subject', 'text', 'type', 'resource_json', 'extraction_timestamp', 'api_url_or_params', 'load_timestamp', 'source_file']


In [2]:
# PARSE THE ORIGINAL FHIR JSON

# Infer the resource schema from the JSON records in Bronze.
resource_schema = spark.read.json(
    bronze_encounter_df.select("resource_json").rdd.map(lambda row: row["resource_json"])
).schema

silver_encounter_df = (
    bronze_encounter_df
    .withColumn(
        "resource",
        from_json(col("resource_json"), resource_schema)
    )
    .select(
        "resource.*",
        "resource_json",
        "extraction_timestamp",
        "api_url_or_params",
        "load_timestamp",
        "source_file"
    )
)

print("FHIR JSON parsed into resource columns.")
silver_encounter_df.printSchema()

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 4, Finished, Available, Finished, False)

FHIR JSON parsed into resource columns.
root
 |-- appointment: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- reference: string (nullable = true)
 |-- class: struct (nullable = true)
 |    |-- code: string (nullable = true)
 |    |-- display: string (nullable = true)
 |    |-- system: string (nullable = true)
 |-- extension: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- url: string (nullable = true)
 |    |    |-- valueBoolean: boolean (nullable = true)
 |    |    |-- valueString: string (nullable = true)
 |-- hospitalization: struct (nullable = true)
 |    |-- admitSource: struct (nullable = true)
 |    |    |-- coding: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- system: string (nullable = true)
 |    |-- dischargeDisposition: str

In [3]:
# CLEAN ENCOUNTER DATA

silver_encounter_df = silver_encounter_df.filter(
    col("id").isNotNull() & (trim(col("id")) != "")
).filter(col("resourceType") == "Encounter")

silver_encounter_df = silver_encounter_df.withColumn(
    "status", lower(trim(col("status")))
)

print("Encounter cleaning completed!")

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 5, Finished, Available, Finished, False)

Encounter cleaning completed!


In [4]:
# DEDUPLICATE BY RESOURCE ID

window_spec = (
    Window
    .partitionBy("id")
    .orderBy(
        col("extraction_timestamp").desc(),
        col("load_timestamp").desc()
    )
)

silver_encounter_df = (
    silver_encounter_df
    .withColumn("row_number", row_number().over(window_spec))
    .filter(col("row_number") == 1)
    .drop("row_number")
)

print("Bronze records:", bronze_encounter_df.count())
print("Silver unique records:", silver_encounter_df.count())

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 6, Finished, Available, Finished, False)

Bronze records: 2005
Silver unique records: 2005


In [5]:
# SAVE SILVER TABLE

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Remove conflicting length field
silver_encounter_df = silver_encounter_df.drop("length")

(
    silver_encounter_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.encounter")
)

print("Saved silver.encounter")
print("Silver records:", spark.table("silver.encounter").count())

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 7, Finished, Available, Finished, False)

Saved silver.encounter
Silver records: 2005


In [6]:
# INITIALIZE SCD TYPE 2 HISTORY IF IT DOES NOT EXIST

history_table = "silver.encounter_history"

if not spark.catalog.tableExists(history_table):
    initial_history_df = (
        spark.table("silver.encounter")
        .withColumn(
            "record_hash",
            sha2(coalesce(col("resource_json"), lit("")), 256)
        )
        .withColumn(
            "effective_start_date",
            col("extraction_timestamp").cast("timestamp")
        )
        .withColumn(
            "effective_end_date",
            lit(None).cast("timestamp")
        )
        .withColumn("is_current", lit(True))
    )

    (
        initial_history_df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(history_table)
    )

    print("Initial SCD Type 2 history created:", history_table)
else:
    print("History table already exists; existing history will be preserved.")

history_df = spark.table(history_table)
print("History records:", history_df.count())
history_df.groupBy("is_current").count().show()

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 8, Finished, Available, Finished, False)

History table already exists; existing history will be preserved.
History records: 2035
+----------+-----+
|is_current|count|
+----------+-----+
|      true| 2035|
+----------+-----+



In [7]:
# DETECT NEW OR CHANGED RECORDS

incoming_df = (
    spark.table("silver.encounter")
    .withColumn(
        "record_hash",
        sha2(coalesce(col("resource_json"), lit("")), 256)
    )
)

current_history_df = (
    spark.table(history_table)
    .filter(col("is_current") == True)
    .select("id", "record_hash")
)

comparison_df = (
    incoming_df.alias("src")
    .join(current_history_df.alias("tgt"), on="id", how="left")
)

changed_or_new_df = comparison_df.filter(
    col("tgt.record_hash").isNull() |
    (col("src.record_hash") != col("tgt.record_hash"))
)

new_versions_df = changed_or_new_df.select("src.*")

print("Incoming records:", incoming_df.count())
print("Existing current records:", current_history_df.count())
print("New or changed records:", new_versions_df.count())

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 9, Finished, Available, Finished, False)

Incoming records: 2005
Existing current records: 2035
New or changed records: 2


In [8]:
# APPLY SCD TYPE 2 CHANGES

changed_ids_df = new_versions_df.select("id").distinct()
changed_count = new_versions_df.count()

if changed_count > 0:
    run_timestamp = spark.sql(
        "SELECT current_timestamp() AS ts"
    ).first()["ts"]

    delta_history = DeltaTable.forName(spark, history_table)

    (
        delta_history.alias("tgt")
        .merge(
            changed_ids_df.alias("src"),
            "tgt.id = src.id AND tgt.is_current = true"
        )
        .whenMatchedUpdate(set={
            "is_current": lit(False),
            "effective_end_date": lit(run_timestamp).cast("timestamp")
        })
        .execute()
    )

    new_history_df = (
        new_versions_df
        .withColumn(
            "effective_start_date",
            lit(run_timestamp).cast("timestamp")
        )
        .withColumn(
            "effective_end_date",
            lit(None).cast("timestamp")
        )
        .withColumn("is_current", lit(True))
    )

    (
        new_history_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(history_table)
    )

    print("SCD Type 2 changes applied:", changed_count)
else:
    print("No new or changed records. History remains unchanged.")

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 10, Finished, Available, Finished, False)

SCD Type 2 changes applied: 2


In [9]:
# VALIDATE HISTORY TABLE

history_df = spark.table(history_table)

print("Total history records:", history_df.count())
history_df.groupBy("is_current").count().show()

duplicate_current_ids = (
    history_df
    .filter(col("is_current") == True)
    .groupBy("id")
    .count()
    .filter(col("count") > 1)
)

print("IDs with more than one current version:", duplicate_current_ids.count())
duplicate_current_ids.show(20, truncate=False)

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 11, Finished, Available, Finished, False)

Total history records: 2037
+----------+-----+
|is_current|count|
+----------+-----+
|      true| 2037|
+----------+-----+

IDs with more than one current version: 0
+---+-----+
|id |count|
+---+-----+
+---+-----+



In [10]:
# FINAL SILVER VALIDATION

spark.table("silver.encounter").select(
    "id", "resourceType", "extraction_timestamp", "load_timestamp"
).show(10, truncate=False)

StatementMeta(, 39d726d5-7e74-4952-a710-a96bc8b28b2c, 12, Finished, Available, Finished, False)

+------------------------------------+------------+--------------------------+--------------------------+
|id                                  |resourceType|extraction_timestamp      |load_timestamp            |
+------------------------------------+------------+--------------------------+--------------------------+
|0193a000-0000-7000-8000-0000000000e4|Encounter   |2026-11-18 18:56:37.567312|2026-11-18 18:57:56.085104|
|380840                              |Encounter   |2026-11-18 18:56:37.567312|2026-11-18 18:57:56.085104|
|463714                              |Encounter   |2026-11-18 18:56:37.567312|2026-11-18 18:57:56.085104|
|492262                              |Encounter   |2026-11-18 18:56:37.567312|2026-11-18 18:57:56.085104|
|492332                              |Encounter   |2026-11-18 18:56:37.567312|2026-11-18 18:57:56.085104|
|793544                              |Encounter   |2026-11-18 18:56:37.567312|2026-11-18 18:57:56.085104|
|793549                              |Encounte